In [6]:
print("Hello world")

Hello world


In [7]:
import pandas as pd 
test_df = pd.read_csv(r"predictions\test_df.csv")
test_df.label = test_df.label.fillna("None")
test_df.head()

,case_name,sent_text,label,hdr_title,hdr_match,hdr_group,y_labelled
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1,Feiten,0
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1,Feiten,0
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1,Beoordeling,1
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1,Beoordeling,1
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1,Beoordeling,0


In [8]:
# ============================================================
# Run all saved models from registry on an existing dataframe
# and append all prediction/probability outputs to the dataframe
# ============================================================

import os
import re
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------

df_model_outputs = test_df.copy()

REGISTRY_PATH = "models\\best_models_registry.json"
TEXT_COL = "sent_text"

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Rows:", len(df_model_outputs))


# ------------------------------------------------------------
# 1. Load model registry
# ------------------------------------------------------------

with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry = json.load(f)

print("Models in registry:")
for k in registry.keys():
    print(" -", k)


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def clean_col_name(x):
    """
    Make labels safe for dataframe column names.
    Example:
    'materiele feiten' -> 'materiele_feiten'
    'None' -> 'none'
    """
    x = str(x).strip().lower()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^a-zA-Z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def resolve_model_path(model_info):
    """
    Prefer saved_path because that should contain the final saved model.
    Fall back to best_model_checkpoint if needed.
    """
    candidates = [model_info.get('saved_path')]

    for c in candidates:
        if c is not None and Path(c).exists():
            return c

    raise FileNotFoundError(
        f"Could not find model path for {model_info.get('model_key')}. "
        f"Tried: {candidates}"
    )


def get_labels(model_key, model_info):
    """
    Get labels in the correct order from the registry.
    """
    if "labels_order" in model_info:
        return model_info["labels_order"]

    if "labels" in model_info:
        return model_info["labels"]

    raise ValueError(f"No labels found for model: {model_key}")


def get_tokenizer(model_path, model_info):
    """
    First try loading tokenizer from saved model folder.
    If tokenizer was not saved there, fall back to model_name from registry.
    """
    try:
        return AutoTokenizer.from_pretrained(model_path)
    except Exception:
        fallback_model_name = model_info.get("model_name", "pdelobelle/robbert-v2-dutch-base")
        print(f"Tokenizer not found in {model_path}. Falling back to {fallback_model_name}")
        return AutoTokenizer.from_pretrained(fallback_model_name)


@torch.no_grad()
def predict_with_model(model_key, model_info, texts):
    """
    Run batched inference for one model.
    Returns:
    - pred_idx
    - pred_label
    - probabilities
    - labels
    """
    model_path = resolve_model_path(model_info)
    labels = get_labels(model_key, model_info)

    max_len = model_info.get("max_len", 256)

    print(f"\nRunning model: {model_key}")
    print(f"Path: {model_path}")
    print(f"Labels: {labels}")

    tokenizer = get_tokenizer(model_path, model_info)

    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    all_probs = []

    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc=model_key):
        batch_texts = texts[start:start + BATCH_SIZE]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        outputs = model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)

        all_probs.append(probs.detach().cpu().numpy())

    probs = np.vstack(all_probs)
    pred_idx = probs.argmax(axis=1)
    pred_label = np.array([labels[i] for i in pred_idx])

    # Clean up GPU memory after each model
    del model
    del tokenizer
    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return pred_idx, pred_label, probs, labels


def add_multiclass_outputs(df_out, prefix, pred_idx, pred_label, probs, labels):
    """
    Add multiclass prediction columns.
    Example:
    text5_pred_idx
    text5_pred_label
    text5_p_none
    text5_p_beoordeling
    ...
    """
    df_out[f"{prefix}_pred_idx"] = pred_idx
    df_out[f"{prefix}_pred_label"] = pred_label

    for i, lab in enumerate(labels):
        lab_clean = clean_col_name(lab)
        df_out[f"{prefix}_p_{lab_clean}"] = probs[:, i]

    return df_out


def add_stage1_outputs(df_out, pred_idx, pred_label, probs, labels):
    """
    Add Stage 1 gatekeeper outputs using your familiar column names:
    stage1_pred_idx
    stage1_pred_label
    stage1_p0
    stage1_p1
    p_none
    p_labelled
    """
    df_out["stage1_pred_idx"] = pred_idx
    df_out["stage1_pred_label"] = pred_label

    # df_out["stage1_p0"] = probs[:, 0]
    # df_out["stage1_p1"] = probs[:, 1]

    # Convenient aliases
    df_out["p_none"] = probs[:, 0]
    df_out["p_labelled"] = probs[:, 1]

    return df_out


def add_ovr_outputs(df_out, model_key, model_info, pred_idx, pred_label, probs, labels):
    """
    Add OvR outputs.

    For ovr_beoordeling, for example:
    ovr_beoordeling_pred_idx
    ovr_beoordeling_pred_label
    ovr_beoordeling_p0
    ovr_beoordeling_p1
    beoordeling_pred
    beoordeling_p_not
    beoordeling_p_pos
    """
    pos_label = model_info["pos_label"]
    pos_clean = clean_col_name(pos_label)
    prefix = f"ovr_{pos_clean}"

    df_out[f"{prefix}_pred_idx"] = pred_idx
    df_out[f"{prefix}_pred_label"] = pred_label
    df_out[f"{prefix}_p0"] = probs[:, 0]
    df_out[f"{prefix}_p1"] = probs[:, 1]

    # Shorter practical columns
    df_out[f"{pos_clean}_pred"] = pred_idx
    df_out[f"{pos_clean}_pred_label"] = pred_label
    df_out[f"{pos_clean}_p_not"] = probs[:, 0]
    df_out[f"{pos_clean}_p_pos"] = probs[:, 1]

    return df_out


# ------------------------------------------------------------
# 3. Basic checks
# ------------------------------------------------------------

if TEXT_COL not in df_model_outputs.columns:
    raise ValueError(f"Text column '{TEXT_COL}' not found in dataframe.")

df_model_outputs[TEXT_COL] = df_model_outputs[TEXT_COL].fillna("").astype(str)
texts = df_model_outputs[TEXT_COL].tolist()


# ------------------------------------------------------------
# 4. Run all models and append outputs
# ------------------------------------------------------------

for model_key, model_info in registry.items():

    pred_idx, pred_label, probs, labels = predict_with_model(
        model_key=model_key,
        model_info=model_info,
        texts=texts
    )

    if model_key == "stage1_gatekeeper":
        df_model_outputs = add_stage1_outputs(
            df_model_outputs,
            pred_idx,
            pred_label,
            probs,
            labels
        )

    elif model_info.get("type") == "ovr_binary":
        df_model_outputs = add_ovr_outputs(
            df_model_outputs,
            model_key,
            model_info,
            pred_idx,
            pred_label,
            probs,
            labels
        )

    elif model_key == "text_only_5way":
        df_model_outputs = add_multiclass_outputs(
            df_model_outputs,
            prefix="text5",
            pred_idx=pred_idx,
            pred_label=pred_label,
            probs=probs,
            labels=labels
        )

    elif model_key == "stage2_text_only_4way":
        df_model_outputs = add_multiclass_outputs(
            df_model_outputs,
            prefix="stage2",
            pred_idx=pred_idx,
            pred_label=pred_label,
            probs=probs,
            labels=labels
        )

    else:
        # Generic fallback for any future model
        safe_prefix = clean_col_name(model_key)
        df_model_outputs = add_multiclass_outputs(
            df_model_outputs,
            prefix=safe_prefix,
            pred_idx=pred_idx,
            pred_label=pred_label,
            probs=probs,
            labels=labels
        )


# ------------------------------------------------------------
# 5. Optional useful combined columns
# ------------------------------------------------------------

# Stage 2 with Stage 1 gate:
# If Stage 1 says not labelled, final prediction is None.
# Otherwise, use the 4-way Stage 2 prediction.
if "stage1_pred_idx" in df_model_outputs.columns and "stage2_pred_label" in df_model_outputs.columns:
    df_model_outputs["pipeline_stage1_stage2_pred"] = np.where(
        df_model_outputs["stage1_pred_idx"] == 0,
        "None",
        df_model_outputs["stage2_pred_label"]
    )

# OvR winner among the four positive probabilities
ovr_pos_cols = [
    "beoordeling_p_pos",
    "beslissing_p_pos",
    "materiele_feiten_p_pos",
    "proceshandelingen_p_pos"
]

if all(c in df_model_outputs.columns for c in ovr_pos_cols):
    ovr_label_map = {
        "beoordeling_p_pos": "beoordeling",
        "beslissing_p_pos": "beslissing",
        "materiele_feiten_p_pos": "materiele feiten",
        "proceshandelingen_p_pos": "proceshandelingen"
    }

    best_ovr_col = df_model_outputs[ovr_pos_cols].idxmax(axis=1)
    df_model_outputs["ovr_best_label"] = best_ovr_col.map(ovr_label_map)
    df_model_outputs["ovr_best_score"] = df_model_outputs[ovr_pos_cols].max(axis=1)

    # Optional gated OvR prediction
    if "stage1_pred_idx" in df_model_outputs.columns:
        df_model_outputs["pipeline_stage1_ovr_pred"] = np.where(
            df_model_outputs["stage1_pred_idx"] == 0,
            "None",
            df_model_outputs["ovr_best_label"]
        )


# ------------------------------------------------------------
# 6. Save output
# ------------------------------------------------------------

output_path = "predictions/df_all_model_outputs.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_model_outputs.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nDone.")
print("Final shape:", df_model_outputs.shape)
print("Saved to:", output_path)

df_model_outputs.head()

Using device: cuda
Rows: 1502
Models in registry:
 - text_only_5way
 - stage1_gatekeeper
 - stage2_text_only_4way
 - ovr_beoordeling
 - ovr_beslissing
 - ovr_materiele_feiten
 - ovr_proceshandelingen

Running model: text_only_5way
Path: models\text_only_5way_best_20260506_122247
Labels: ['None', 'beoordeling', 'beslissing', 'materiele feiten', 'proceshandelingen']


text_only_5way:   0%|          | 0/47 [00:00<?, ?it/s]


Running model: stage1_gatekeeper
Path: models\stage1_gatekeeper_best_20260506_140445
Labels: ['not_labelled', 'labelled']


stage1_gatekeeper:   0%|          | 0/47 [00:00<?, ?it/s]


Running model: stage2_text_only_4way
Path: models\stage2_text_only_4way_best_20260506_141904
Labels: ['beoordeling', 'beslissing', 'materiele feiten', 'proceshandelingen']


...

ovr_materiele_feiten:   0%|          | 0/47 [00:00<?, ?it/s]


Running model: ovr_proceshandelingen
Path: models\ovr_proceshandelingen_best_20260506_160617
Labels: ['not_proceshandelingen', 'proceshandelingen']


ovr_proceshandelingen:   0%|          | 0/47 [00:00<?, ?it/s]


Done.
Final shape: (1502, 60)
Saved to: predictions/df_all_model_outputs.csv


,case_name,sent_text,label,hdr_title,hdr_match,hdr_group,y_labelled,text5_pred_idx,text5_pred_label,text5_p_none,...,ovr_proceshandelingen_p0,ovr_proceshandelingen_p1,proceshandelingen_pred,proceshandelingen_pred_label,proceshandelingen_p_not,proceshandelingen_p_pos,pipeline_stage1_stage2_pred,ovr_best_label,ovr_best_score,pipeline_stage1_ovr_pred
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1,Feiten,0,1,beoordeling,0.393809,...,0.998965,0.001035,0,not_proceshandelingen,0.998965,0.001035,beoordeling,beoordeling,0.989040,beoordeling
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1,Feiten,0,1,beoordeling,0.040119,...,0.994490,0.005510,0,not_proceshandelingen,0.994490,0.005510,beoordeling,beoordeling,0.998138,beoordeling
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1,Beoordeling,1,3,materiele feiten,0.011060,...,0.998819,0.001181,0,not_proceshandelingen,0.998819,0.001181,materiele feiten,materiele feiten,0.999256,materiele feiten
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1,Beoordeling,1,3,materiele feiten,0.042123,...,0.998702,0.001298,0,not_proceshandelingen,0.998702,0.001298,materiele feiten,materiele feiten,0.998631,materiele feiten
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1,Beoordeling,0,0,None,0.989501,...,0.992466,0.007534,0,not_proceshandelingen,0.992466,0.007534,None,materiele feiten,0.084251,None


## 5 Way Classification

In [9]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# 5-way classification evaluation
# ------------------------------------------------------------

LABELS_5WAY = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

y_true = df_model_outputs["label"]
y_pred = df_model_outputs["text5_pred_label"]

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

acc = accuracy_score(y_true, y_pred)
print("5-way classification accuracy:", round(acc, 4))

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report - 5-way model")
print(
    classification_report(
        y_true,
        y_pred,
        labels=LABELS_5WAY,
        target_names=LABELS_5WAY,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS_5WAY
)
print("\nRaw Confusion Matrix (5-way model)")
print(cm)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in LABELS_5WAY],
    columns=[f"pred_{label}" for label in LABELS_5WAY]
)

print("\nConfusion Matrix - 5-way model")
display(cm_df)

5-way classification accuracy: 0.6458

Classification Report - 5-way model
                   precision    recall  f1-score   support

             None     0.6748    0.7172    0.6953       541
      beoordeling     0.6262    0.6761    0.6502       389
       beslissing     0.8966    0.7536    0.8189        69
 materiele feiten     0.5942    0.5522    0.5724       297
proceshandelingen     0.5954    0.5000    0.5435       206

         accuracy                         0.6458      1502
        macro avg     0.6774    0.6398    0.6561      1502
     weighted avg     0.6456    0.6458    0.6442      1502


Raw Confusion Matrix (5-way model)
[[388  85   5  45  18]
 [ 67 263   1  33  25]
 [ 11   3  52   3   0]
 [ 81  25   0 164  27]
 [ 28  44   0  31 103]]

Confusion Matrix - 5-way model


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,388,85,5,45,18
true_beoordeling,67,263,1,33,25
true_beslissing,11,3,52,3,0
true_materiele feiten,81,25,0,164,27
true_proceshandelingen,28,44,0,31,103


## OVR Models 

In [10]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# OvR evaluation on Stage 2 legal-only subset
# Excludes gold label == "None"
# ------------------------------------------------------------

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

ovr_settings = {
    "beoordeling": "beoordeling_pred",
    "beslissing": "beslissing_pred",
    "materiele feiten": "materiele_feiten_pred",
    "proceshandelingen": "proceshandelingen_pred",
}

# Only evaluate OvR models on sentences that truly belong to one of the 4 legal classes
ovr_eval_df = df_model_outputs[df_model_outputs["label"].isin(LEGAL_LABELS)].copy()

print("Total rows in full dataframe:", len(df_model_outputs))
print("Rows used for OvR Stage 2 evaluation:", len(ovr_eval_df))
print("Excluded None rows:", len(df_model_outputs) - len(ovr_eval_df))


# ------------------------------------------------------------
# Evaluate each OvR model separately
# ------------------------------------------------------------

for target_label, pred_col in ovr_settings.items():
    
    print("\n" + "=" * 80)
    print(f"OvR evaluation for: {target_label}")
    print("=" * 80)
    
    if pred_col not in ovr_eval_df.columns:
        raise ValueError(f"Prediction column not found: {pred_col}")
    
    # Binary gold labels:
    # 1 = current target label
    # 0 = any other legal label
    y_true = (ovr_eval_df["label"] == target_label).astype(int)
    
    # Binary model prediction:
    # 1 = predicted as target label
    # 0 = predicted as not target label
    y_pred = ovr_eval_df[pred_col].astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy for {target_label}: {acc:.4f}")
    
    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=[f"not_{target_label}", target_label],
            digits=4,
            zero_division=0
        )
    )
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_not_{target_label}", f"true_{target_label}"],
        columns=[f"pred_not_{target_label}", f"pred_{target_label}"]
    )
    
    print("Confusion Matrix")
    display(cm_df)

Total rows in full dataframe: 1502
Rows used for OvR Stage 2 evaluation: 961
Excluded None rows: 541

OvR evaluation for: beoordeling

Accuracy for beoordeling: 0.8117

Classification Report
                 precision    recall  f1-score   support

not_beoordeling     0.8510    0.8287    0.8397       572
    beoordeling     0.7574    0.7866    0.7718       389

       accuracy                         0.8117       961
      macro avg     0.8042    0.8077    0.8057       961
   weighted avg     0.8131    0.8117    0.8122       961

Confusion Matrix


,pred_not_beoordeling,pred_beoordeling
true_not_beoordeling,474,98
true_beoordeling,83,306



OvR evaluation for: beslissing

Accuracy for beslissing: 0.9834

Classification Report
                precision    recall  f1-score   support

not_beslissing     0.9834    0.9989    0.9911       892
    beslissing     0.9818    0.7826    0.8710        69

      accuracy                         0.9834       961
     macro avg     0.9826    0.8907    0.9310       961
  weighted avg     0.9833    0.9834    0.9825       961

Confusion Matrix


,pred_not_beslissing,pred_beslissing
true_not_beslissing,891,1
true_beslissing,15,54



OvR evaluation for: materiele feiten

Accuracy for materiele feiten: 0.8127

Classification Report
                      precision    recall  f1-score   support

not_materiele feiten     0.8315    0.9142    0.8709       664
    materiele feiten     0.7532    0.5859    0.6591       297

            accuracy                         0.8127       961
           macro avg     0.7924    0.7500    0.7650       961
        weighted avg     0.8073    0.8127    0.8054       961

Confusion Matrix


,pred_not_materiele feiten,pred_materiele feiten
true_not_materiele feiten,607,57
true_materiele feiten,123,174



OvR evaluation for: proceshandelingen

Accuracy for proceshandelingen: 0.8345

Classification Report
                       precision    recall  f1-score   support

not_proceshandelingen     0.8850    0.9073    0.8960       755
    proceshandelingen     0.6257    0.5680    0.5954       206

             accuracy                         0.8345       961
            macro avg     0.7553    0.7376    0.7457       961
         weighted avg     0.8294    0.8345    0.8316       961

Confusion Matrix


,pred_not_proceshandelingen,pred_proceshandelingen
true_not_proceshandelingen,685,70
true_proceshandelingen,89,117


## Stage 1: Binary Gatekeeper

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# Stage 1 gatekeeper evaluation
# Binary: None vs labelled
# ------------------------------------------------------------

STAGE1_LABELS = [0, 1]
STAGE1_NAMES = ["not_labelled / None", "labelled"]

y_true = df_model_outputs["y_labelled"].astype(int)
y_pred = df_model_outputs["stage1_pred_idx"].astype(int)

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

acc = accuracy_score(y_true, y_pred)
print("Stage 1 gatekeeper accuracy:", round(acc, 4))

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report - Stage 1 gatekeeper")
print(
    classification_report(
        y_true,
        y_pred,
        labels=STAGE1_LABELS,
        target_names=STAGE1_NAMES,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=STAGE1_LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=["true_not_labelled_None", "true_labelled"],
    columns=["pred_not_labelled_None", "pred_labelled"]
)

print("\nConfusion Matrix - Stage 1 gatekeeper")
display(cm_df)

Stage 1 gatekeeper accuracy: 0.7783

Classification Report - Stage 1 gatekeeper
                     precision    recall  f1-score   support

not_labelled / None     0.6933    0.6895    0.6914       541
           labelled     0.8257    0.8283    0.8270       961

           accuracy                         0.7783      1502
          macro avg     0.7595    0.7589    0.7592      1502
       weighted avg     0.7780    0.7783    0.7782      1502


Confusion Matrix - Stage 1 gatekeeper


,pred_not_labelled_None,pred_labelled
true_not_labelled_None,373,168
true_labelled,165,796


## Stage 2: 4 way (Without None)

In [12]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# Stage 2 4-way evaluation
# Only evaluate on gold labelled / legal-role sentences
# ------------------------------------------------------------

STAGE2_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

# Keep only sentences that truly belong to one of the 4 legal labels
stage2_eval_df = df_model_outputs[
    df_model_outputs["label"].isin(STAGE2_LABELS)
].copy()

print("Total rows in full dataframe:", len(df_model_outputs))
print("Rows used for Stage 2 4-way evaluation:", len(stage2_eval_df))
print("Excluded None rows:", len(df_model_outputs) - len(stage2_eval_df))

# ------------------------------------------------------------
# Gold and predicted labels
# ------------------------------------------------------------

y_true = stage2_eval_df["label"]
y_pred = stage2_eval_df["stage2_pred_label"]

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

acc = accuracy_score(y_true, y_pred)
print("\nStage 2 4-way accuracy:", round(acc, 4))

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report - Stage 2 4-way model")
print(
    classification_report(
        y_true,
        y_pred,
        labels=STAGE2_LABELS,
        target_names=STAGE2_LABELS,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=STAGE2_LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in STAGE2_LABELS],
    columns=[f"pred_{label}" for label in STAGE2_LABELS]
)

print("\nConfusion Matrix - Stage 2 4-way model")
display(cm_df)

Total rows in full dataframe: 1502
Rows used for Stage 2 4-way evaluation: 961
Excluded None rows: 541

Stage 2 4-way accuracy: 0.7138

Classification Report - Stage 2 4-way model
                   precision    recall  f1-score   support

      beoordeling     0.7471    0.8278    0.7854       389
       beslissing     0.9483    0.7971    0.8661        69
 materiele feiten     0.7273    0.6195    0.6691       297
proceshandelingen     0.5708    0.6068    0.5882       206

         accuracy                         0.7138       961
        macro avg     0.7484    0.7128    0.7272       961
     weighted avg     0.7176    0.7138    0.7130       961


Confusion Matrix - Stage 2 4-way model


,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_beoordeling,322,3,38,26
true_beslissing,8,55,2,4
true_materiele feiten,49,0,184,64
true_proceshandelingen,52,0,29,125


## Bayesian Update

In [13]:
import numpy as np
import pandas as pd
import re
import os

# ------------------------------------------------------------
# 0. Setup
# ------------------------------------------------------------

robbert_bayes_df = df_model_outputs.copy()
train_df = pd.read_csv(r"predictions\train_df.csv")
train_df["label"] = train_df["label"].fillna("None").astype(str).str.strip()
robbert_bayes_df["label"] = robbert_bayes_df["label"].fillna("None").astype(str).str.strip()

LAM = 3.0
ALPHA = 1.0

ALL_LABELS = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def clean_col_name(x):
    x = str(x).strip().lower()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^a-zA-Z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    """
    Estimate P(label | hdr_group) from training data only.
    """
    counts = (
        df_train
        .groupby(["hdr_group", label_col])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )

    probs = counts + alpha
    probs = probs.div(probs.sum(axis=1), axis=0)

    return probs


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    """
    Fallback prior for unseen header groups.
    """
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    prior = counts / counts.sum()
    return prior.values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    """
    Create metadata prior matrix for the dataframe being predicted.
    """
    def get_prior_for_row(hdr_group):
        if pd.notna(hdr_group) and hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    meta_prior_mat = np.vstack(
        df_apply["hdr_group"].apply(get_prior_for_row)
    )

    return pd.DataFrame(
        meta_prior_mat,
        columns=labels,
        index=df_apply.index
    )


def combine_log_scores(text_probs_df, meta_prior_df, lam=1.0, eps=1e-12):
    """
    Bayesian-style log fusion:
    log score = log P(label | text) + lambda * log P(label | header)
    """
    text = np.clip(text_probs_df.values, eps, 1.0)
    meta = np.clip(meta_prior_df.values, eps, 1.0)

    log_score = np.log(text) + lam * np.log(meta)

    # numerical stability
    log_score = log_score - log_score.max(axis=1, keepdims=True)

    score = np.exp(log_score)
    score = score / score.sum(axis=1, keepdims=True)

    return pd.DataFrame(
        score,
        columns=text_probs_df.columns,
        index=text_probs_df.index
    )


def add_bayes_prob_columns(df, prefix, bayes_probs, labels):
    """
    Add Bayesian-updated probability columns and prediction label.
    """
    for lab in labels:
        clean_lab = clean_col_name(lab)
        df[f"{prefix}_p_{clean_lab}"] = bayes_probs[lab]

    df[f"{prefix}_pred_label"] = bayes_probs.idxmax(axis=1)

    return df

In [14]:
# ------------------------------------------------------------
# RobBERT 5-way Bayesian update
# ------------------------------------------------------------

robbert_5way_prob_cols = {
    "None": "text5_p_none",
    "beoordeling": "text5_p_beoordeling",
    "beslissing": "text5_p_beslissing",
    "materiele feiten": "text5_p_materiele_feiten",
    "proceshandelingen": "text5_p_proceshandelingen"
}

robbert_5way_text_probs = robbert_bayes_df[
    [robbert_5way_prob_cols[label] for label in ALL_LABELS]
].copy()

robbert_5way_text_probs.columns = ALL_LABELS

header_prior_5way = make_header_prior(
    df_train=train_df,
    label_col="label",
    labels=ALL_LABELS,
    alpha=ALPHA
)

global_prior_5way = make_global_prior(
    df_train=train_df,
    label_col="label",
    labels=ALL_LABELS,
    alpha=ALPHA
)

meta_prior_5way = get_meta_prior_df(
    df_apply=robbert_bayes_df,
    header_prior_train=header_prior_5way,
    global_prior=global_prior_5way,
    labels=ALL_LABELS
)

robbert_bayes_5way_probs = combine_log_scores(
    text_probs_df=robbert_5way_text_probs,
    meta_prior_df=meta_prior_5way,
    lam=LAM
)

robbert_bayes_df = add_bayes_prob_columns(
    df=robbert_bayes_df,
    prefix="robbert_bayes_5way",
    bayes_probs=robbert_bayes_5way_probs,
    labels=ALL_LABELS
)

print("Done: RobBERT 5-way Bayesian update")

robbert_bayes_df[
    ["label", "text5_pred_label", "robbert_bayes_5way_pred_label"]
].head()

Done: RobBERT 5-way Bayesian update


,label,text5_pred_label,robbert_bayes_5way_pred_label
0,None,beoordeling,None
1,None,beoordeling,None
2,materiele feiten,materiele feiten,materiele feiten
3,materiele feiten,materiele feiten,materiele feiten
4,None,None,None


In [15]:
robbert_bayes_df.columns

Index(['case_name', 'sent_text', 'label', 'hdr_title', 'hdr_match',
       'hdr_group', 'y_labelled', 'text5_pred_idx', 'text5_pred_label',
       'text5_p_none', 'text5_p_beoordeling', 'text5_p_beslissing',
       'text5_p_materiele_feiten', 'text5_p_proceshandelingen',
       'stage1_pred_idx', 'stage1_pred_label', 'p_none', 'p_labelled',
       'stage2_pred_idx', 'stage2_pred_label', 'stage2_p_beoordeling',
       'stage2_p_beslissing', 'stage2_p_materiele_feiten',
       'stage2_p_proceshandelingen', 'ovr_beoordeling_pred_idx',
       'ovr_beoordeling_pred_label', 'ovr_beoordeling_p0',
       'ovr_beoordeling_p1', 'beoordeling_pred', 'beoordeling_pred_label',
       'beoordeling_p_not', 'beoordeling_p_pos', 'ovr_beslissing_pred_idx',
       'ovr_beslissing_pred_label', 'ovr_beslissing_p0', 'ovr_beslissing_p1',
       'beslissing_pred', 'beslissing_pred_label', 'beslissing_p_not',
       'beslissing_p_pos', 'ovr_materiele_feiten_pred_idx',
       'ovr_materiele_feiten_pred_label', '

In [16]:
# ------------------------------------------------------------
# RobBERT Stage 1 Bayesian update
# Binary: 0 = None / not labelled, 1 = labelled
# ------------------------------------------------------------

stage1_train = train_df.copy()

stage1_train["y_stage1"] = np.where(
    stage1_train["label"] == "None",
    0,
    1
)

STAGE1_LABELS = [0, 1]

robbert_stage1_text_probs = robbert_bayes_df[
    ["p_none", "p_labelled"]
].copy()

robbert_stage1_text_probs.columns = STAGE1_LABELS

header_prior_stage1 = make_header_prior(
    df_train=stage1_train,
    label_col="y_stage1",
    labels=STAGE1_LABELS,
    alpha=ALPHA
)

global_prior_stage1 = make_global_prior(
    df_train=stage1_train,
    label_col="y_stage1",
    labels=STAGE1_LABELS,
    alpha=ALPHA
)

meta_prior_stage1 = get_meta_prior_df(
    df_apply=robbert_bayes_df,
    header_prior_train=header_prior_stage1,
    global_prior=global_prior_stage1,
    labels=STAGE1_LABELS
)

robbert_bayes_stage1_probs = combine_log_scores(
    text_probs_df=robbert_stage1_text_probs,
    meta_prior_df=meta_prior_stage1,
    lam=LAM
)

robbert_bayes_df["robbert_bayes_stage1_p0"] = robbert_bayes_stage1_probs[0]
robbert_bayes_df["robbert_bayes_stage1_p1"] = robbert_bayes_stage1_probs[1]

robbert_bayes_df["robbert_bayes_stage1_pred_idx"] = robbert_bayes_stage1_probs.idxmax(axis=1)

robbert_bayes_df["robbert_bayes_stage1_pred_label"] = np.where(
    robbert_bayes_df["robbert_bayes_stage1_pred_idx"] == 0,
    "not_labelled",
    "labelled"
)

print("Done: RobBERT Stage 1 Bayesian update")

robbert_bayes_df[
    ["y_labelled", "stage1_pred_idx", "robbert_bayes_stage1_pred_idx", "p_labelled", "robbert_bayes_stage1_p1"]
].head()

Done: RobBERT Stage 1 Bayesian update


,y_labelled,stage1_pred_idx,robbert_bayes_stage1_pred_idx,p_labelled,robbert_bayes_stage1_p1
0,0,1,1,0.997465,0.994311
1,0,1,1,0.999780,0.999504
2,1,1,1,0.999867,0.999991
3,1,1,1,0.999871,0.999991
4,0,0,0,0.000262,0.003799


In [17]:
# ------------------------------------------------------------
# RobBERT Stage 2 4-way Bayesian update
# ------------------------------------------------------------

stage2_train = train_df[
    train_df["label"].isin(LEGAL_LABELS)
].copy()

robbert_stage2_prob_cols = {
    "beoordeling": "stage2_p_beoordeling",
    "beslissing": "stage2_p_beslissing",
    "materiele feiten": "stage2_p_materiele_feiten",
    "proceshandelingen": "stage2_p_proceshandelingen"
}

robbert_stage2_text_probs = robbert_bayes_df[
    [robbert_stage2_prob_cols[label] for label in LEGAL_LABELS]
].copy()

robbert_stage2_text_probs.columns = LEGAL_LABELS

header_prior_stage2 = make_header_prior(
    df_train=stage2_train,
    label_col="label",
    labels=LEGAL_LABELS,
    alpha=ALPHA
)

global_prior_stage2 = make_global_prior(
    df_train=stage2_train,
    label_col="label",
    labels=LEGAL_LABELS,
    alpha=ALPHA
)

meta_prior_stage2 = get_meta_prior_df(
    df_apply=robbert_bayes_df,
    header_prior_train=header_prior_stage2,
    global_prior=global_prior_stage2,
    labels=LEGAL_LABELS
)

robbert_bayes_stage2_probs = combine_log_scores(
    text_probs_df=robbert_stage2_text_probs,
    meta_prior_df=meta_prior_stage2,
    lam=LAM
)

robbert_bayes_df = add_bayes_prob_columns(
    df=robbert_bayes_df,
    prefix="robbert_bayes_stage2",
    bayes_probs=robbert_bayes_stage2_probs,
    labels=LEGAL_LABELS
)

print("Done: RobBERT Stage 2 Bayesian update")

robbert_bayes_df[
    ["label", "stage2_pred_label", "robbert_bayes_stage2_pred_label"]
].head()

Done: RobBERT Stage 2 Bayesian update


,label,stage2_pred_label,robbert_bayes_stage2_pred_label
0,None,beoordeling,materiele feiten
1,None,beoordeling,materiele feiten
2,materiele feiten,materiele feiten,materiele feiten
3,materiele feiten,materiele feiten,materiele feiten
4,None,beslissing,beoordeling


In [18]:
# ------------------------------------------------------------
# RobBERT OvR Bayesian update
# ------------------------------------------------------------

ovr_train_base = train_df[
    train_df["label"].isin(LEGAL_LABELS)
].copy()

for target_label in LEGAL_LABELS:

    clean_label = clean_col_name(target_label)

    print(f"Running RobBERT OvR Bayesian update for: {target_label}")

    ovr_train = ovr_train_base.copy()

    ovr_train[f"y_{clean_label}"] = np.where(
        ovr_train["label"] == target_label,
        1,
        0
    )

    binary_labels = [0, 1]

    text_probs = robbert_bayes_df[
        [f"{clean_label}_p_not", f"{clean_label}_p_pos"]
    ].copy()

    text_probs.columns = binary_labels

    header_prior_ovr = make_header_prior(
        df_train=ovr_train,
        label_col=f"y_{clean_label}",
        labels=binary_labels,
        alpha=ALPHA
    )

    global_prior_ovr = make_global_prior(
        df_train=ovr_train,
        label_col=f"y_{clean_label}",
        labels=binary_labels,
        alpha=ALPHA
    )

    meta_prior_ovr = get_meta_prior_df(
        df_apply=robbert_bayes_df,
        header_prior_train=header_prior_ovr,
        global_prior=global_prior_ovr,
        labels=binary_labels
    )

    bayes_probs = combine_log_scores(
        text_probs_df=text_probs,
        meta_prior_df=meta_prior_ovr,
        lam=LAM
    )

    robbert_bayes_df[f"robbert_bayes_{clean_label}_p_not"] = bayes_probs[0]
    robbert_bayes_df[f"robbert_bayes_{clean_label}_p_pos"] = bayes_probs[1]
    robbert_bayes_df[f"robbert_bayes_{clean_label}_pred"] = bayes_probs.idxmax(axis=1)

print("Done: RobBERT OvR Bayesian update")

Running RobBERT OvR Bayesian update for: beoordeling
Running RobBERT OvR Bayesian update for: beslissing
Running RobBERT OvR Bayesian update for: materiele feiten
Running RobBERT OvR Bayesian update for: proceshandelingen
Done: RobBERT OvR Bayesian update


In [19]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

def evaluate_model(df, y_true_col, y_pred_col, labels, title):
    """
    Print accuracy, classification report, and confusion matrix.
    """

    y_true = df[y_true_col]
    y_pred = df[y_pred_col]

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy: {acc:.4f}")

    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=[str(x) for x in labels],
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels]
    )

    print("\nConfusion Matrix")
    display(cm_df)

In [20]:
# ------------------------------------------------------------
# 1. RobBERT + Bayes 5-way classification
# ------------------------------------------------------------

evaluate_model(
    df=robbert_bayes_df,
    y_true_col="label",
    y_pred_col="robbert_bayes_5way_pred_label",
    labels=ALL_LABELS,
    title="RobBERT + Bayes: 5-way classification"
)


RobBERT + Bayes: 5-way classification

Accuracy: 0.6525

Classification Report
                   precision    recall  f1-score   support

             None     0.5863    0.7726    0.6667       541
      beoordeling     0.6454    0.7018    0.6724       389
       beslissing     0.9302    0.5797    0.7143        69
 materiele feiten     0.7707    0.4074    0.5330       297
proceshandelingen     0.7711    0.6214    0.6882       206

         accuracy                         0.6525      1502
        macro avg     0.7407    0.6166    0.6549      1502
     weighted avg     0.6792    0.6525    0.6469      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,418,86,2,17,18
true_beoordeling,84,273,1,13,18
true_beslissing,28,1,40,0,0
true_materiele feiten,149,25,0,121,2
true_proceshandelingen,34,38,0,6,128


In [21]:
# ------------------------------------------------------------
# 2. RobBERT + Bayes Stage 1 gatekeeper
# Binary: 0 = None / not labelled, 1 = labelled
# ------------------------------------------------------------

evaluate_model(
    df=robbert_bayes_df,
    y_true_col="y_labelled",
    y_pred_col="robbert_bayes_stage1_pred_idx",
    labels=[0, 1],
    title="RobBERT + Bayes: Stage 1 gatekeeper"
)


RobBERT + Bayes: Stage 1 gatekeeper

Accuracy: 0.7790

Classification Report
              precision    recall  f1-score   support

           0     0.7103    0.6525    0.6802       541
           1     0.8129    0.8502    0.8311       961

    accuracy                         0.7790      1502
   macro avg     0.7616    0.7513    0.7556      1502
weighted avg     0.7760    0.7790    0.7768      1502


Confusion Matrix


,pred_0,pred_1
true_0,353,188
true_1,144,817


In [22]:
# ------------------------------------------------------------
# 3. RobBERT + Bayes Stage 2 4-way classification
# Evaluate only on gold legal-labelled sentences
# ------------------------------------------------------------

stage2_eval_df = robbert_bayes_df[
    robbert_bayes_df["label"].isin(LEGAL_LABELS)
].copy()

print("Total rows:", len(robbert_bayes_df))
print("Rows used for Stage 2 evaluation:", len(stage2_eval_df))
print("Excluded None rows:", len(robbert_bayes_df) - len(stage2_eval_df))

evaluate_model(
    df=stage2_eval_df,
    y_true_col="label",
    y_pred_col="robbert_bayes_stage2_pred_label",
    labels=LEGAL_LABELS,
    title="RobBERT + Bayes: Stage 2 4-way classification"
)

Total rows: 1502
Rows used for Stage 2 evaluation: 961
Excluded None rows: 541

RobBERT + Bayes: Stage 2 4-way classification

Accuracy: 0.7846

Classification Report
                   precision    recall  f1-score   support

      beoordeling     0.7445    0.8689    0.8019       389
       beslissing     0.9825    0.8116    0.8889        69
 materiele feiten     0.8352    0.7508    0.7908       297
proceshandelingen     0.7486    0.6650    0.7044       206

         accuracy                         0.7846       961
        macro avg     0.8277    0.7741    0.7965       961
     weighted avg     0.7905    0.7846    0.7838       961


Confusion Matrix


,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_beoordeling,338,1,30,20
true_beslissing,8,56,2,3
true_materiele feiten,51,0,223,23
true_proceshandelingen,57,0,12,137


In [23]:
# ------------------------------------------------------------
# 4. RobBERT + Bayes OvR evaluation
# Each binary OvR model evaluated on gold legal-labelled subset
# ------------------------------------------------------------

ovr_eval_df = robbert_bayes_df[
    robbert_bayes_df["label"].isin(LEGAL_LABELS)
].copy()

print("Total rows:", len(robbert_bayes_df))
print("Rows used for OvR evaluation:", len(ovr_eval_df))
print("Excluded None rows:", len(robbert_bayes_df) - len(ovr_eval_df))

ovr_settings = {
    "beoordeling": "robbert_bayes_beoordeling_pred",
    "beslissing": "robbert_bayes_beslissing_pred",
    "materiele feiten": "robbert_bayes_materiele_feiten_pred",
    "proceshandelingen": "robbert_bayes_proceshandelingen_pred"
}

for target_label, pred_col in ovr_settings.items():

    temp_df = ovr_eval_df.copy()

    temp_df[f"gold_{clean_col_name(target_label)}"] = (
        temp_df["label"] == target_label
    ).astype(int)

    evaluate_model(
        df=temp_df,
        y_true_col=f"gold_{clean_col_name(target_label)}",
        y_pred_col=pred_col,
        labels=[0, 1],
        title=f"RobBERT + Bayes: OvR binary model for {target_label}"
    )

Total rows: 1502
Rows used for OvR evaluation: 961
Excluded None rows: 541

RobBERT + Bayes: OvR binary model for beoordeling

Accuracy: 0.8429

Classification Report
              precision    recall  f1-score   support

           0     0.8526    0.8899    0.8708       572
           1     0.8269    0.7738    0.7995       389

    accuracy                         0.8429       961
   macro avg     0.8398    0.8318    0.8351       961
weighted avg     0.8422    0.8429    0.8419       961


Confusion Matrix


,pred_0,pred_1
true_0,509,63
true_1,88,301



RobBERT + Bayes: OvR binary model for beslissing

Accuracy: 0.9802

Classification Report
              precision    recall  f1-score   support

           0     0.9802    0.9989    0.9895       892
           1     0.9808    0.7391    0.8430        69

    accuracy                         0.9802       961
   macro avg     0.9805    0.8690    0.9162       961
weighted avg     0.9802    0.9802    0.9789       961


Confusion Matrix


,pred_0,pred_1
true_0,891,1
true_1,18,51



RobBERT + Bayes: OvR binary model for materiele feiten

Accuracy: 0.8398

Classification Report
              precision    recall  f1-score   support

           0     0.8236    0.9774    0.8939       664
           1     0.9133    0.5320    0.6723       297

    accuracy                         0.8398       961
   macro avg     0.8684    0.7547    0.7831       961
weighted avg     0.8513    0.8398    0.8255       961


Confusion Matrix


,pred_0,pred_1
true_0,649,15
true_1,139,158



RobBERT + Bayes: OvR binary model for proceshandelingen

Accuracy: 0.8730

Classification Report
              precision    recall  f1-score   support

           0     0.8754    0.9775    0.9237       755
           1     0.8559    0.4903    0.6235       206

    accuracy                         0.8730       961
   macro avg     0.8657    0.7339    0.7736       961
weighted avg     0.8713    0.8730    0.8593       961


Confusion Matrix


,pred_0,pred_1
true_0,738,17
true_1,105,101


Pipeline Evaluations

In [24]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ALL_LABELS = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]


def make_pipeline_A_probabilistic(
    df,
    stage1_p_none_col,
    stage1_p_labelled_col,
    stage2_prob_cols,
    prefix
):
    """
    Pipeline A probabilistic combination.

    Final 5-way probabilities:
        P(None) = P(Stage 1 = None)
        P(legal label) = P(Stage 1 = labelled) * P(Stage 2 = legal label | labelled)
    """

    df = df.copy()

    p_none = df[stage1_p_none_col].astype(float).values
    p_labelled = df[stage1_p_labelled_col].astype(float).values

    df[f"{prefix}_p_none"] = p_none

    for label in LEGAL_LABELS:
        clean_label = label.replace(" ", "_")
        df[f"{prefix}_p_{clean_label}"] = (
            p_labelled * df[stage2_prob_cols[label]].astype(float).values
        )

    final_prob_cols = [
        f"{prefix}_p_none",
        f"{prefix}_p_beoordeling",
        f"{prefix}_p_beslissing",
        f"{prefix}_p_materiele_feiten",
        f"{prefix}_p_proceshandelingen"
    ]

    probs_for_pred = df[final_prob_cols].copy()
    probs_for_pred.columns = ALL_LABELS

    df[f"{prefix}_pred"] = probs_for_pred.idxmax(axis=1)

    return df


def evaluate_5way_prediction(df, pred_col, title):
    y_true = df["label"].fillna("None").astype(str).str.strip()
    y_pred = df[pred_col].fillna("None").astype(str).str.strip()

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy: {acc:.4f}")

    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=ALL_LABELS,
            target_names=ALL_LABELS,
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=ALL_LABELS)

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{label}" for label in ALL_LABELS],
        columns=[f"pred_{label}" for label in ALL_LABELS]
    )

    print("\nConfusion Matrix")
    display(cm_df)

In [25]:
robbert_stage2_prob_cols = {
    "beoordeling": "stage2_p_beoordeling",
    "beslissing": "stage2_p_beslissing",
    "materiele feiten": "stage2_p_materiele_feiten",
    "proceshandelingen": "stage2_p_proceshandelingen"
}

df_model_outputs = make_pipeline_A_probabilistic(
    df=df_model_outputs,
    stage1_p_none_col="p_none",
    stage1_p_labelled_col="p_labelled",
    stage2_prob_cols=robbert_stage2_prob_cols,
    prefix="robbert_pipeline_A_prob"
)

evaluate_5way_prediction(
    df=df_model_outputs,
    pred_col="robbert_pipeline_A_prob_pred",
    title="RobBERT Pipeline A: Probabilistic Stage 1 + Stage 2"
)


RobBERT Pipeline A: Probabilistic Stage 1 + Stage 2

Accuracy: 0.6258

Classification Report
                   precision    recall  f1-score   support

             None     0.6895    0.6895    0.6895       541
      beoordeling     0.5953    0.7147    0.6495       389
       beslissing     0.8367    0.5942    0.6949        69
 materiele feiten     0.5880    0.4613    0.5170       297
proceshandelingen     0.5236    0.5388    0.5311       206

         accuracy                         0.6258      1502
        macro avg     0.6466    0.5997    0.6164      1502
     weighted avg     0.6290    0.6258    0.6235      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,373,103,6,33,26
true_beoordeling,53,278,2,33,23
true_beslissing,17,5,41,2,4
true_materiele feiten,78,34,0,137,48
true_proceshandelingen,20,47,0,28,111


In [26]:
robbert_bayes_stage2_prob_cols = {
    "beoordeling": "robbert_bayes_stage2_p_beoordeling",
    "beslissing": "robbert_bayes_stage2_p_beslissing",
    "materiele feiten": "robbert_bayes_stage2_p_materiele_feiten",
    "proceshandelingen": "robbert_bayes_stage2_p_proceshandelingen"
}

robbert_bayes_df = make_pipeline_A_probabilistic(
    df=robbert_bayes_df,
    stage1_p_none_col="robbert_bayes_stage1_p0",
    stage1_p_labelled_col="robbert_bayes_stage1_p1",
    stage2_prob_cols=robbert_bayes_stage2_prob_cols,
    prefix="robbert_bayes_pipeline_A_prob"
)

evaluate_5way_prediction(
    df=robbert_bayes_df,
    pred_col="robbert_bayes_pipeline_A_prob_pred",
    title="RobBERT + Bayes Pipeline A: Probabilistic Stage 1 + Stage 2"
)


RobBERT + Bayes Pipeline A: Probabilistic Stage 1 + Stage 2

Accuracy: 0.6605

Classification Report
                   precision    recall  f1-score   support

             None     0.7100    0.6562    0.6820       541
      beoordeling     0.5919    0.7866    0.6755       389
       beslissing     0.9000    0.6522    0.7563        69
 materiele feiten     0.6532    0.5455    0.5945       297
proceshandelingen     0.6631    0.6019    0.6310       206

         accuracy                         0.6605      1502
        macro avg     0.7036    0.6485    0.6679      1502
     weighted avg     0.6705    0.6605    0.6595      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,355,109,4,48,25
true_beoordeling,39,306,1,25,18
true_beslissing,15,5,45,2,2
true_materiele feiten,75,42,0,162,18
true_proceshandelingen,16,55,0,11,124


## Pipeline B

In [27]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ALL_LABELS = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]


def make_pipeline_B_probabilistic(
    df,
    stage1_p_none_col,
    stage1_p_labelled_col,
    ovr_pos_cols,
    legal_labels,
    prefix,
    eps=1e-12
):
    """
    Pipeline B probabilistic combination.

    Step 1:
        Use Stage 1 probabilities:
        P(None)
        P(labelled)

    Step 2:
        Normalize the four OvR positive probabilities across legal labels:
        q_k = p_ovr_k / sum(p_ovr_all)

    Step 3:
        Construct final 5-way probabilities:
        P(None) = stage1_p_none
        P(k)    = stage1_p_labelled * q_k

    This creates a proper 5-way probability distribution.
    """

    df = df.copy()

    # Stage 1 probabilities
    p_none = df[stage1_p_none_col].astype(float).values
    p_labelled = df[stage1_p_labelled_col].astype(float).values

    # OvR positive probabilities
    ovr_scores = df[ovr_pos_cols].astype(float).values
    ovr_scores = np.clip(ovr_scores, eps, 1.0)

    # Normalize OvR probabilities across the four legal labels
    ovr_norm = ovr_scores / ovr_scores.sum(axis=1, keepdims=True)

    # Final 5-way probabilities
    df[f"{prefix}_p_none"] = p_none

    for i, label in enumerate(legal_labels):
        clean_label = label.replace(" ", "_")
        df[f"{prefix}_p_{clean_label}"] = p_labelled * ovr_norm[:, i]

    final_prob_cols = [
        f"{prefix}_p_none",
        f"{prefix}_p_beoordeling",
        f"{prefix}_p_beslissing",
        f"{prefix}_p_materiele_feiten",
        f"{prefix}_p_proceshandelingen"
    ]

    # Rename temporarily so idxmax gives clean labels
    probs_for_pred = df[final_prob_cols].copy()
    probs_for_pred.columns = ALL_LABELS

    df[f"{prefix}_pred"] = probs_for_pred.idxmax(axis=1)

    return df


def evaluate_5way_prediction(df, pred_col, title):
    y_true = df["label"].fillna("None").astype(str).str.strip()
    y_pred = df[pred_col].fillna("None").astype(str).str.strip()

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy: {acc:.4f}")

    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=ALL_LABELS,
            target_names=ALL_LABELS,
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=ALL_LABELS
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{label}" for label in ALL_LABELS],
        columns=[f"pred_{label}" for label in ALL_LABELS]
    )

    print("\nConfusion Matrix")
    display(cm_df)

In [28]:
df_model_outputs.columns

Index(['case_name', 'sent_text', 'label', 'hdr_title', 'hdr_match',
       'hdr_group', 'y_labelled', 'text5_pred_idx', 'text5_pred_label',
       'text5_p_none', 'text5_p_beoordeling', 'text5_p_beslissing',
       'text5_p_materiele_feiten', 'text5_p_proceshandelingen',
       'stage1_pred_idx', 'stage1_pred_label', 'p_none', 'p_labelled',
       'stage2_pred_idx', 'stage2_pred_label', 'stage2_p_beoordeling',
       'stage2_p_beslissing', 'stage2_p_materiele_feiten',
       'stage2_p_proceshandelingen', 'ovr_beoordeling_pred_idx',
       'ovr_beoordeling_pred_label', 'ovr_beoordeling_p0',
       'ovr_beoordeling_p1', 'beoordeling_pred', 'beoordeling_pred_label',
       'beoordeling_p_not', 'beoordeling_p_pos', 'ovr_beslissing_pred_idx',
       'ovr_beslissing_pred_label', 'ovr_beslissing_p0', 'ovr_beslissing_p1',
       'beslissing_pred', 'beslissing_pred_label', 'beslissing_p_not',
       'beslissing_p_pos', 'ovr_materiele_feiten_pred_idx',
       'ovr_materiele_feiten_pred_label', '

In [29]:
robbert_ovr_pos_cols = [
    "beoordeling_p_pos",
    "beslissing_p_pos",
    "materiele_feiten_p_pos",
    "proceshandelingen_p_pos"
]

df_model_outputs = make_pipeline_B_probabilistic(
    df=df_model_outputs,
    stage1_p_none_col="p_none",
    stage1_p_labelled_col="p_labelled",
    ovr_pos_cols=robbert_ovr_pos_cols,
    legal_labels=LEGAL_LABELS,
    prefix="robbert_pipeline_B"
)

evaluate_5way_prediction(
    df=df_model_outputs,
    pred_col="robbert_pipeline_B_pred",
    title="RobBERT Pipeline B: Stage 1 probability + normalized OvR probabilities"
)


RobBERT Pipeline B: Stage 1 probability + normalized OvR probabilities

Accuracy: 0.6272

Classification Report
                   precision    recall  f1-score   support

             None     0.6882    0.6895    0.6888       541
      beoordeling     0.5876    0.7069    0.6418       389
       beslissing     0.8571    0.6087    0.7119        69
 materiele feiten     0.6076    0.4848    0.5393       297
proceshandelingen     0.5243    0.5243    0.5243       206

         accuracy                         0.6272      1502
        macro avg     0.6530    0.6028    0.6212      1502
     weighted avg     0.6315    0.6272    0.6256      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,373,101,5,33,29
true_beoordeling,53,275,1,32,28
true_beslissing,18,6,42,3,0
true_materiele feiten,78,33,1,144,41
true_proceshandelingen,20,53,0,25,108


In [30]:
robbert_bayes_ovr_pos_cols = [
    "robbert_bayes_beoordeling_p_pos",
    "robbert_bayes_beslissing_p_pos",
    "robbert_bayes_materiele_feiten_p_pos",
    "robbert_bayes_proceshandelingen_p_pos"
]

robbert_bayes_df = make_pipeline_B_probabilistic(
    df=robbert_bayes_df,
    stage1_p_none_col="robbert_bayes_stage1_p0",
    stage1_p_labelled_col="robbert_bayes_stage1_p1",
    ovr_pos_cols=robbert_bayes_ovr_pos_cols,
    legal_labels=LEGAL_LABELS,
    prefix="robbert_bayes_pipeline_B"
)

evaluate_5way_prediction(
    df=robbert_bayes_df,
    pred_col="robbert_bayes_pipeline_B_pred",
    title="RobBERT + Bayes Pipeline B: Stage 1 probability + normalized OvR probabilities"
)


RobBERT + Bayes Pipeline B: Stage 1 probability + normalized OvR probabilities

Accuracy: 0.6678

Classification Report
                   precision    recall  f1-score   support

             None     0.7080    0.6543    0.6801       541
      beoordeling     0.5988    0.7789    0.6771       389
       beslissing     0.8913    0.5942    0.7130        69
 materiele feiten     0.6733    0.5690    0.6168       297
proceshandelingen     0.6834    0.6602    0.6716       206

         accuracy                         0.6678      1502
        macro avg     0.7110    0.6513    0.6717      1502
     weighted avg     0.6779    0.6678    0.6672      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,354,112,4,46,25
true_beoordeling,39,303,1,24,22
true_beslissing,17,7,41,2,2
true_materiele feiten,74,40,0,169,14
true_proceshandelingen,16,44,0,10,136
